# Tutorial 2: Feature Engineering

## Overview

This notebook demonstrates how to transform raw aligned data into features for regime identification.

### Feature Architecture

```
Asset Features (Layer C):          Macro Features (Layers A/B):
├─ Downside Deviation (LOG scale)  ├─ Stock-Bond Correlation
├─ Sortino Ratios                  ├─ VIX / Volatility Proxy
├─ EWM Returns                     ├─ Yield Curve Slope
├─ Realized Volatility             ├─ Credit Spreads
├─ Cumulative Returns              └─ Policy Uncertainty
├─ Skewness
├─ Maximum Drawdown
└─ Current Return
```

### Learning Objectives

1. Understand asset-specific vs. macro features
2. Compute 21 features per asset
3. Learn critical LOG-scale transformation for downside deviation
4. Generate macro features for regime classification
5. Visualize feature distributions and relationships

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 30)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Imports successful")

## 1. Load Data

In [ ]:
from src.core.data import DataPipeline

# Load aligned daily data
pipeline = DataPipeline(mode='basic')
raw_data = pipeline.load('1985-01-01', '2010-12-31')

print(f"✓ Loaded raw data: {raw_data.shape}")

## 2. Engineer Features

The `engineer_features()` function computes:
- **Asset features**: 21 features per asset (Layer C)
- **Macro features**: 5-75 features depending on mode (Layers A/B)

In [ ]:
from src.core.features import engineer_features

# Engineer features (basic mode)
asset_features, macro_features = engineer_features(raw_data, complexity='basic')

print(f"\n✓ Feature engineering complete")
print(f"  Assets: {len(asset_features)}")
print(f"  Asset features per asset: {list(asset_features.values())[0].shape[1] if len(asset_features) > 0 else 0}")
print(f"  Macro features: {macro_features.shape[1]}")

## 3. Explore Asset Features (Layer C)

Each asset has 21 features computed at multiple time horizons.

In [ ]:
# Examine features for one asset
asset_name = list(asset_features.keys())[0] if len(asset_features) > 0 else None

if asset_name:
    asset_df = asset_features[asset_name]
    
    print(f"\nFeatures for {asset_name}:")
    print(f"  Shape: {asset_df.shape}")
    print(f"  Date range: {asset_df.index.min()} to {asset_df.index.max()}")
    print(f"\nFeature list:")
    for i, col in enumerate(asset_df.columns, 1):
        print(f"  {i:2d}. {col}")
    
    # Show sample data
    print(f"\nSample data (first 5 rows):")
    display(asset_df.head())
    
    # Summary statistics
    print(f"\nSummary statistics:")
    display(asset_df.describe())
else:
    print("⚠ No asset features available")

### Critical: Downside Deviation (LOG Scale)

⚠️ **Important**: Downside deviation is computed on LOG scale to emphasize risk dynamics.

Formula:
```
DD_log = log(sqrt(EWM(downside_returns^2)))
```

This transformation:
- Stabilizes variance across regimes
- Emphasizes relative changes in risk
- Improves regime separation

In [ ]:
if asset_name:
    # Plot downside deviation across horizons
    dd_cols = [c for c in asset_df.columns if c.startswith('dd_')]
    
    if len(dd_cols) > 0:
        fig, axes = plt.subplots(len(dd_cols), 1, figsize=(14, 3*len(dd_cols)))
        
        if len(dd_cols) == 1:
            axes = [axes]
        
        for i, col in enumerate(dd_cols):
            ax = axes[i]
            asset_df[col].plot(ax=ax, color='darkred', linewidth=1)
            ax.set_title(f'{asset_name}: {col} (LOG scale)', fontweight='bold')
            ax.set_ylabel('Log(Downside Dev)')
            ax.grid(alpha=0.3)
            
            # Highlight high-risk periods (top 10%)
            threshold = asset_df[col].quantile(0.90)
            high_risk = asset_df[col] > threshold
            ax.fill_between(asset_df.index, asset_df[col].min(), asset_df[col].max(),
                           where=high_risk, alpha=0.2, color='red', label='High Risk')
            ax.legend()
        
        plt.tight_layout()
        plt.show()
        
        print("\n📊 Notice how LOG scale compresses large values and expands small values")
        print("   This helps the regime model detect subtle shifts in risk.")

### Sortino Ratio

Sortino ratio measures return relative to downside risk (better than Sharpe for asymmetric returns).

In [ ]:
if asset_name:
    # Plot Sortino ratios
    sortino_cols = [c for c in asset_df.columns if c.startswith('sortino_')]
    
    if len(sortino_cols) > 0:
        fig, ax = plt.subplots(figsize=(14, 6))
        
        for col in sortino_cols:
            asset_df[col].plot(ax=ax, label=col, linewidth=1, alpha=0.7)
        
        ax.set_title(f'{asset_name}: Sortino Ratios (Multiple Horizons)', fontweight='bold')
        ax.set_ylabel('Sortino Ratio')
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
        ax.grid(alpha=0.3)
        ax.legend()
        
        plt.tight_layout()
        plt.show()
        
        # Statistics
        print(f"\nSortino Ratio Statistics:")
        for col in sortino_cols:
            median_sortino = asset_df[col].median()
            pct_positive = (asset_df[col] > 0).sum() / len(asset_df) * 100
            print(f"  {col}: median={median_sortino:.2f}, positive={pct_positive:.1f}%")

### Feature Correlation Analysis

Understand relationships between features.

In [ ]:
if asset_name:
    # Compute correlation matrix
    corr_matrix = asset_df.corr()
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(14, 12))
    
    sns.heatmap(
        corr_matrix,
        cmap='RdBu_r',
        center=0,
        vmin=-1,
        vmax=1,
        square=True,
        linewidths=0.5,
        cbar_kws={'label': 'Correlation'},
        ax=ax
    )
    
    ax.set_title(f'{asset_name}: Feature Correlation Matrix', fontweight='bold', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    # Find highly correlated pairs
    print("\nHighly Correlated Feature Pairs (|r| > 0.8):")
    high_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.8:
                high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))
    
    for feat1, feat2, corr in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
        print(f"  {feat1} <-> {feat2}: {corr:.3f}")

## 4. Explore Macro Features (Layers A/B)

Macro features capture market-wide conditions for regime identification.

In [ ]:
print(f"Macro Features ({macro_features.shape[1]} total):")
for i, col in enumerate(macro_features.columns, 1):
    print(f"  {i}. {col}")

# Summary statistics
print(f"\nMacro Feature Summary:")
display(macro_features.describe())

### Stock-Bond Correlation

Rolling 252-day correlation between stocks and bonds - key regime indicator.

In [ ]:
if 'stock_bond_corr' in macro_features.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    macro_features['stock_bond_corr'].plot(ax=ax, color='purple', linewidth=1.5)
    ax.set_title('Stock-Bond Correlation (252-day rolling)', fontweight='bold')
    ax.set_ylabel('Correlation')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axhline(0.5, color='green', linewidth=0.8, linestyle=':', alpha=0.5, label='High Positive')
    ax.axhline(-0.5, color='red', linewidth=0.8, linestyle=':', alpha=0.5, label='High Negative')
    ax.grid(alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Regime interpretation
    print("\n📊 Stock-Bond Correlation Regimes:")
    print("  • Positive (> 0.3): Risk-on environment")
    print("  • Near-zero (-0.3 to 0.3): Normal diversification")
    print("  • Negative (< -0.3): Flight-to-quality / Crisis")

### Yield Curve Slope

10Y - 3M spread - classic recession indicator.

In [ ]:
if 'yield_slope' in macro_features.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    macro_features['yield_slope'].plot(ax=ax, color='darkgreen', linewidth=1.5)
    ax.set_title('Yield Curve Slope (10Y - 3M)', fontweight='bold')
    ax.set_ylabel('Spread (bps)')
    ax.axhline(0, color='red', linewidth=1, linestyle='--', label='Inversion (Recession Signal)')
    ax.grid(alpha=0.3)
    ax.legend()
    
    # Shade inversions
    inversions = macro_features['yield_slope'] < 0
    ax.fill_between(macro_features.index, 
                    macro_features['yield_slope'].min(), 
                    macro_features['yield_slope'].max(),
                    where=inversions, 
                    alpha=0.2, 
                    color='red',
                    label='Inverted')
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    inversion_days = inversions.sum()
    total_days = len(macro_features)
    print(f"\n📊 Yield Curve Statistics:")
    print(f"  Mean slope: {macro_features['yield_slope'].mean():.2f} bps")
    print(f"  Inverted days: {inversion_days} ({inversion_days/total_days*100:.1f}%)")

### Policy Uncertainty (EPU)

Economic Policy Uncertainty Index - captures political/policy risk.

In [ ]:
if 'policy_uncertainty' in macro_features.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    macro_features['policy_uncertainty'].plot(ax=ax, color='orange', linewidth=1.5)
    ax.set_title('Economic Policy Uncertainty (Normalized)', fontweight='bold')
    ax.set_ylabel('Std. Deviations from Mean')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axhline(2, color='red', linewidth=0.8, linestyle=':', alpha=0.5, label='High Uncertainty')
    ax.grid(alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    plt.show()

## 5. Cross-Asset Feature Comparison

Compare features across all assets.

In [ ]:
# Compare downside deviation across assets
if len(asset_features) > 1:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for asset_name, features_df in asset_features.items():
        if 'dd_63d' in features_df.columns:
            features_df['dd_63d'].plot(ax=ax, label=asset_name, linewidth=1.5, alpha=0.7)
    
    ax.set_title('Downside Deviation (63d) - All Assets', fontweight='bold')
    ax.set_ylabel('Log(Downside Dev)')
    ax.grid(alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Asset Risk Comparison:")
    print("  Higher values = Higher downside risk")
    print("  Divergence = Different regime states across assets")

## 6. Feature Distributions

Understand feature distributions for modeling.

In [ ]:
if asset_name:
    # Select key features to plot
    key_features = ['dd_63d', 'sortino_63d', 'ewm_return_63d', 'volatility_63d']
    available_features = [f for f in key_features if f in asset_df.columns]
    
    if len(available_features) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes = axes.flatten()
        
        for i, feature in enumerate(available_features[:4]):
            ax = axes[i]
            
            # Histogram with KDE
            data_clean = asset_df[feature].dropna()
            data_clean.hist(bins=50, ax=ax, alpha=0.6, color='steelblue', edgecolor='black')
            
            # Overlay KDE
            ax2 = ax.twinx()
            data_clean.plot.kde(ax=ax2, color='red', linewidth=2)
            ax2.set_ylabel('Density', color='red')
            ax2.tick_params(axis='y', labelcolor='red')
            
            ax.set_title(f'{asset_name}: {feature}', fontweight='bold')
            ax.set_xlabel('Value')
            ax.set_ylabel('Frequency')
            ax.grid(alpha=0.3)
        
        plt.tight_layout()
        plt.show()

## 7. Feature Engineering Summary

### Asset Features (21 per asset):

| Category | Features | Horizons |
|----------|----------|----------|
| **Risk** | Downside Deviation (LOG) | 21d, 63d, 126d |
| **Risk-Adjusted Return** | Sortino Ratio | 21d, 63d, 126d |
| **Momentum** | EWM Returns | 21d, 63d, 126d |
| **Volatility** | Realized Volatility | 21d, 63d, 126d |
| **Performance** | Cumulative Returns | 21d, 63d, 126d |
| **Shape** | Skewness | 63d |
| **Drawdown** | Maximum Drawdown | 126d |
| **Current** | Return | 1d |

### Macro Features (5-75 depending on mode):

| Category | Features |
|----------|----------|
| **Correlation** | Stock-Bond Correlation |
| **Volatility** | VIX Proxy (GPR) |
| **Term Structure** | Yield Curve Slope |
| **Credit** | Credit Spread Proxy |
| **Policy** | Economic Policy Uncertainty |

### Critical Transformations:

1. ⚠️ **Downside Deviation**: LOG scale (emphasizes risk dynamics)
2. **EWM**: Half-lives of 21, 63, 126 days (1mo, 3mo, 6mo)
3. **Normalization**: Z-scores for policy indices

### Next Steps:

→ **Tutorial 3**: Regime Identification - Use features to identify bull/bear regimes

### Resources:

- `src/core/features.py`: Feature engineering implementation
- Theory: See `DATA_PREPROCESSING_GUIDE.md`